# Dynamic Time Wraping

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('modified_data/wdiEDA.csv', index_col=0)


In [4]:
df

,Country,year,carbon_int_gdp,energy_int_gdp,energy_pc,gdp_pc,res_share,ghg_pc
0,Albania,1990,3.765618,2.948070,0.813445,-0.963894,-2.138792,1.530003
1,Albania,1991,3.217255,2.773759,-0.621923,-1.315289,-1.053111,0.139315
2,Albania,1992,0.733200,1.446898,-1.543458,-1.376436,0.944542,-1.524612
3,Albania,1993,0.245119,0.943823,-1.578781,-1.288237,1.567000,-1.479929
4,Albania,1994,0.167542,0.868089,-1.405809,-1.202992,1.610427,-1.041313
...,...,...,...,...,...,...,...,...
4411,Zimbabwe,2017,-1.443628,-1.290788,-0.855089,0.094159,0.980628,-1.032894
4412,Zimbabwe,2018,-1.153521,-1.195499,-0.720029,0.275185,0.653900,-0.829470
4413,Zimbabwe,2019,-1.240643,-0.962748,-0.730221,-0.146303,0.838572,-1.081907
4414,Zimbabwe,2020,-1.596608,-0.991301,-0.940888,-0.612274,1.278945,-1.344476


In [5]:
countries = sorted(df["Country"].unique())
features = ["carbon_int_gdp", "energy_int_gdp", "energy_pc", "gdp_pc", "res_share", "ghg_pc"]

X = np.stack([
    df[df.Country == c].sort_values("year")[features].to_numpy()
    for c in countries
])  # shape (N, T, F)

In [6]:
X.shape

(138, 32, 6)

In [ ]:
from tslearn.metrics import cdist_dtw

D = cdist_dtw(
    X,
    global_constraint="sakoe_chiba",
    sakoe_chiba_radius= 10,
)

print(X.shape)                      # (N, T, F), 
print(D.shape)                      # (N, N)
print(np.isnan(X).any())            # False
print(np.allclose(D, D.T))          # True
print(np.diag(D).max())             # 0.0
print(np.isfinite(D).all())         # False = okno za wąskie albo błąd w danych

dtw_df = pd.DataFrame(D, index=countries, columns=countries)


(138, 32, 6)
(138, 138)
False
True
0.0
True


In [12]:
dtw_df

,Albania,Algeria,Angola,Argentina,Armenia,Australia,Austria,Azerbaijan,Bahrain,Bangladesh,...,Ukraine,United Arab Emirates,United Kingdom,United States,Uruguay,Uzbekistan,"Venezuela, RB",Viet Nam,Zambia,Zimbabwe
Albania,0.000000,15.679534,17.136875,11.424139,7.978174,15.218078,14.490943,14.393795,16.763990,13.254809,...,17.846879,23.593454,18.034672,18.225144,14.660924,16.160791,21.718962,13.110821,12.354543,19.248077
Algeria,15.679534,0.000000,16.046169,17.114915,16.007339,21.519549,20.296433,16.326996,18.746969,15.498293,...,21.187192,21.573430,22.090812,21.647545,16.421637,20.100179,19.946216,14.359956,12.104024,21.749112
Angola,17.136875,16.046169,0.000000,14.189941,16.298401,13.249718,15.536597,14.260267,13.877045,17.625387,...,14.753594,20.401262,14.348159,14.503686,18.826086,13.440531,17.674276,18.281188,15.324917,18.194352
Argentina,11.424139,17.114915,14.189941,0.000000,15.270647,12.422403,12.492491,17.216010,15.333414,15.129342,...,17.955306,21.588867,16.486539,16.202735,13.210641,16.114164,17.628491,15.201697,16.467047,19.143561
Armenia,7.978174,16.007339,16.298401,15.270647,0.000000,16.219837,15.010956,10.636861,17.107295,17.520875,...,14.074546,21.464273,15.335855,15.963712,16.167174,13.437236,21.537868,17.009623,11.365446,16.014594
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Uzbekistan,16.160791,20.100179,13.440531,16.114164,13.437236,11.899377,13.814231,11.151981,12.647561,19.931464,...,10.223801,19.523447,10.041729,10.072564,18.730108,0.000000,20.399005,20.195540,14.672724,14.328850
"Venezuela, RB",21.718962,19.946216,17.674276,17.628491,21.537868,17.953858,18.681705,20.985664,17.990874,22.681694,...,17.631790,12.410754,17.961226,18.270932,19.206052,20.399005,0.000000,21.631003,21.739137,18.919613
Viet Nam,13.110821,14.359956,18.281188,15.201697,17.009623,19.624112,18.794475,19.856936,19.280684,5.146205,...,22.889749,24.055767,22.343712,22.222987,16.683018,20.195540,21.631003,0.000000,15.009192,23.466726
Zambia,12.354543,12.104024,15.324917,16.467047,11.365446,18.842809,17.588646,10.434018,14.331139,14.833915,...,16.339723,22.320324,17.102796,16.371857,18.419246,14.672724,21.739137,15.009192,0.000000,15.823456
